In [1]:
# Imports
import sys
from pathlib import Path
import warnings
import os
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="X does not have valid feature names")

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.tabsyn.models import TabSyn

# ---------------- Dataset paths (NURSERY) ----------------
dataset_path = ROOT / "raw_data" / "nursery.csv"
output_path  = ROOT / "discretized_data" / "nursery.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Only discretize once
if not output_path.exists():
    discretize_preprocess(str(dataset_path), str(output_path))
else:
    print("✅ Discretized file already exists:", output_path)

# ---------------- Pipeline paths ----------------
input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / "nursery")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "nursery" / "tabsyn")

# ---------------- Helper: ensure TabSyn .npy files exist ----------------
def ensure_tabsyn_npy(data_dir: str):
    for split in ["train", "test"]:
        y_csv = os.path.join(data_dir, f"y_{split}.csv")
        y_npy = os.path.join(data_dir, f"y_{split}.npy")

        x_csv = os.path.join(data_dir, f"x_{split}.csv")
        xcat_npy = os.path.join(data_dir, f"X_cat_{split}.npy")

        if os.path.exists(y_csv) and not os.path.exists(y_npy):
            y = pd.read_csv(y_csv).iloc[:, 0].astype(int).to_numpy()
            np.save(y_npy, y, allow_pickle=True)

        if os.path.exists(x_csv) and not os.path.exists(xcat_npy):
            X = pd.read_csv(x_csv).astype(str).to_numpy()
            np.save(xcat_npy, X, allow_pickle=True)

# ---------------- Model + Pipeline (THIS is where params apply) ----------------
pipeline = TrainTestSplitPipeline(
    model=lambda: TabSyn(
        d_token=64,
        decoder_epochs=60,
        diffusion_epochs=60,
        diffusion_steps=600,
        decoder_batch_size=128,
        diffusion_batch_size=128,
        lr=1e-3,
        weight_decay=5e-4,
        patience=20,
        seed=42,
    )
)

# ---------------- Run (SMALL for CPU) ----------------
try:
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
        size_category="small",  
    )
    print(result)

except FileNotFoundError:
    ensure_tabsyn_npy(output_dir)

    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
        size_category="small",   # ✅ changed to small
    )
    print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
✅ Discretized file already exists: C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)


[TabSyn] Synthetic data saved:
  X -> C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\tabsyn\x_synth.csv
  y -> C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\tabsyn\y_synth.csv


C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:10:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results\nursery\tabsyn_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.3769
F1 Score: 0.3464

MLP:
Accuracy: 0.3603
F1 Score: 0.3524

RF:
Accuracy: 0.3349
F1 Score: 0.3327

XGBoost:
Accuracy: 0.3306
F1 Score: 0.3268
Train test split pipeline executed successfully.
